# Project19 Database Viewer

This notebook is for looking at the database visually: annotated images and Alpamayo prediction graphs.

## Setup

In [ ]:
import importlib
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
pipeline_dir = cwd if cwd.name == "pipeline" else cwd / "pipeline"
sys.path.insert(0, str(pipeline_dir))

import notebook_helpers
importlib.reload(notebook_helpers)
from notebook_helpers import open_explorer

db = open_explorer()

## Annotated Images

In [ ]:
db.show_first_frame()

In [ ]:
# Leave SOURCE as None to show the first frames in the database.
# Set it to something like "route_1_segment_00" to focus one segment.
SOURCE = None

db.show_gallery(source=SOURCE, limit=6)

## Pedestrian Annotations

In [ ]:
PEDESTRIAN_LIMIT = 6

pedestrian_rows = db.conn.execute(
    """
    SELECT DISTINCT f.id
    FROM frames f
    JOIN annotations a ON a.frame_id = f.id
    JOIN label_categories lc ON lc.annotation_id = a.id
    WHERE lc.category = 'pedestrian'
      AND lc.present = 1
    ORDER BY f.source, f.frame_number
    LIMIT ?
    """,
    (PEDESTRIAN_LIMIT,),
).fetchall()

if not pedestrian_rows:
    print("No pedestrian annotations found.")

for row in pedestrian_rows:
    db.show_frame(int(row["id"]), figsize=(10, 5))

## Latest Prediction

In [ ]:
latest = db.conn.execute(
    "SELECT id FROM alpamayo_predictions ORDER BY id DESC LIMIT 1"
).fetchone()

if latest is None:
    print("No predictions found.")
else:
    db.plot_prediction(int(latest["id"]))

## Left/Right Turn Predictions

In [ ]:
TURN_LIMIT = 3

for turn_word in ["left", "right"]:
    turn_rows = db.conn.execute(
        """
        SELECT ap.id
        FROM alpamayo_predictions ap
        JOIN frames f ON f.id = ap.frame_id
        WHERE LOWER(ap.nav_command) LIKE ?
        ORDER BY f.source, f.frame_number, ap.id
        LIMIT ?
        """,
        (f"%{turn_word}%", TURN_LIMIT),
    ).fetchall()

    if not turn_rows:
        print(f"No {turn_word} predictions found.")

    for row in turn_rows:
        db.plot_prediction(int(row["id"]))